# Atividade 8 — Limpeza e Preparação da Base
### Projeto: Recomendação de Treinos Personalizados em Academia

Este notebook aplica, sobre a base `dataset_academia.csv`, o conjunto de tratamentos
apontado como necessário no diagnóstico de qualidade da Atividade 5 (Encontro 5).

O arquivo original (`dataset_academia.csv`) **não é alterado**: todas as transformações
são aplicadas sobre uma cópia, salva ao final como `dataset_academia_tratado.csv`.


## 1. Carregamento dos dados

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 160)

df_original = pd.read_csv("dataset_academia.csv")

print(f"Linhas: {df_original.shape[0]}  |  Colunas: {df_original.shape[1]}")
df_original.head()


Linhas: 300  |  Colunas: 8


,aluno_id,idade,objetivo,nivel_experiencia,dias_disponiveis_semana,restricao_fisica,equipamento_disponivel,treino_recomendado
0,1,32,Emagrecimento,Iniciante,5,Nenhuma,Limitado,Treino D
1,2,37,Hipertrofia,Intermediario,4,Joelho,Completo,Treino D
2,3,24,Condicionamento,Iniciante,2,Ombro,Limitado,Treino D
3,4,31,Emagrecimento,Iniciante,5,Nenhuma,Limitado,Treino D
4,5,23,Hipertrofia,Intermediario,2,Ombro,Completo,Treino B


In [2]:
df_original.dtypes


aluno_id                   int64
idade                      int64
objetivo                     str
nivel_experiencia            str
dias_disponiveis_semana    int64
restricao_fisica             str
equipamento_disponivel       str
treino_recomendado           str
dtype: object

## 2. Recapitulação do diagnóstico (Atividade 5)

Principais achados já registrados no diagnóstico de qualidade:

| Eixo | Resultado |
|---|---|
| Dados ausentes | 0 em todas as colunas |
| Duplicados | 0 linhas idênticas; 0 `aluno_id` repetido |
| Inconsistências | `nivel_experiencia` sem acento; variáveis ordinais armazenadas como texto solto; dicionário dizia idade "16–65", base vai até 51; 8% de ruído de rótulo |
| Desbalanceamento | leve — Treino D mais frequente, Treino B menos frequente |
| Viés | grupo "sem equipamento" tem rótulo quase sempre fixo em Treino D |

As células abaixo confirmam esses números na base atual antes de qualquer tratamento.


In [3]:
print("Valores ausentes por coluna:")
print(df_original.isna().sum())

print("\nLinhas duplicadas (todas as colunas):", df_original.duplicated().sum())
print("aluno_id duplicado:", df_original["aluno_id"].duplicated().sum())


Valores ausentes por coluna:
aluno_id                   0
idade                      0
objetivo                   0
nivel_experiencia          0
dias_disponiveis_semana    0
restricao_fisica           0
equipamento_disponivel     0
treino_recomendado         0
dtype: int64

Linhas duplicadas (todas as colunas): 0
aluno_id duplicado: 0


In [4]:
print("Faixa de idade observada:", df_original["idade"].min(), "a", df_original["idade"].max())
print("Dicionário de dados (Atividade 4) registrava 16 a 65 anos.")


Faixa de idade observada: 16 a 50
Dicionário de dados (Atividade 4) registrava 16 a 65 anos.


In [5]:
print(df_original["treino_recomendado"].value_counts())
print()
print(df_original["treino_recomendado"].value_counts(normalize=True).round(3) * 100)


treino_recomendado
Treino D    88
Treino C    77
Treino A    74
Treino B    61
Name: count, dtype: int64

treino_recomendado
Treino D    29.3
Treino C    25.7
Treino A    24.7
Treino B    20.3
Name: proportion, dtype: float64


## 3. Transformações aplicadas

Cada transformação abaixo corresponde a um item do diagnóstico da Atividade 5.
As justificativas foram revisadas e consolidadas a partir do feedback recebido em aula
(prioridade para transformações que não distorcem os dados nem tentam "corrigir"
informação que não temos como validar).


### 3.1 Remover `aluno_id` das features

**Motivo:** é apenas um identificador sequencial de geração da base, sem relação com o
perfil do aluno. Mantê-lo como feature faria o modelo aprender a posição do registro,
não o perfil da pessoa — um risco de vazamento indireto. Ele é preservado apenas como
índice do DataFrame, para rastreabilidade, e removido das colunas de trabalho.

**Estratégia alternativa considerada:** manter `aluno_id` e simplesmente não usá-lo no
treinamento. Descartada porque deixar a coluna no arquivo tratado aumenta o risco de
alguém incluí-la por engano como feature em uma etapa futura.


In [6]:
df = df_original.copy()
df = df.set_index("aluno_id")

print("Colunas após remoção do aluno_id como feature:")
print(list(df.columns))


Colunas após remoção do aluno_id como feature:
['idade', 'objetivo', 'nivel_experiencia', 'dias_disponiveis_semana', 'restricao_fisica', 'equipamento_disponivel', 'treino_recomendado']


### 3.2 Padronizar nomenclatura das variáveis categóricas

**Motivo:** o diagnóstico apontou inconsistência de nomenclatura (ex.: `nivel_experiencia`
sem acento nos valores, possíveis espaços/maiúsculas inconsistentes). Aplicamos
`strip()` e padronização de capitalização em todas as colunas de texto, para garantir
que valores como `"iniciante"`, `"Iniciante "` e `"Iniciante"` não sejam tratados como
categorias diferentes mais adiante.


In [7]:
colunas_texto = ["objetivo", "nivel_experiencia", "restricao_fisica",
                  "equipamento_disponivel", "treino_recomendado"]

for c in colunas_texto:
    antes = df[c].nunique()
    df[c] = df[c].str.strip().str.title()
    depois = df[c].nunique()
    print(f"{c}: {antes} categorias antes -> {depois} categorias depois")


objetivo: 3 categorias antes -> 3 categorias depois
nivel_experiencia: 3 categorias antes -> 3 categorias depois
restricao_fisica: 4 categorias antes -> 4 categorias depois
equipamento_disponivel: 3 categorias antes -> 3 categorias depois
treino_recomendado: 4 categorias antes -> 4 categorias depois


### 3.3 Converter variáveis ordinais para categoria ordenada

**Motivo:** o diagnóstico identificou que `nivel_experiencia` e `equipamento_disponivel`
são ordinais (têm uma ordem natural de progressão), mas estavam armazenadas como texto
nominal solto, sem essa ordem registrada nos dados. Convertemos para `pd.Categorical`
com `ordered=True`, o que permite comparações (`>`, `<`) corretas mais adiante e evita
que codificações futuras (Encontro 9) tratem, por engano, essas colunas como nominais
sem ordem (o que exigiria one-hot em vez de encoding ordinal).

**Estratégia alternativa considerada:** deixar como texto e só definir a ordem no
momento da modelagem. Descartada porque a ordem é uma propriedade do dado, não do
modelo — registrá-la aqui evita retrabalho e erro de interpretação no Encontro 9.


In [8]:
ordem_experiencia = ["Iniciante", "Intermediario", "Avancado"]
ordem_equipamento = ["Nenhum", "Limitado", "Completo"]

df["nivel_experiencia"] = pd.Categorical(df["nivel_experiencia"],
                                          categories=ordem_experiencia, ordered=True)
df["equipamento_disponivel"] = pd.Categorical(df["equipamento_disponivel"],
                                               categories=ordem_equipamento, ordered=True)

print(df["nivel_experiencia"].dtype)
print(df["equipamento_disponivel"].dtype)


category
category


### 3.4 Confirmar ausência de valores nulos e duplicados

**Motivo:** o diagnóstico já indicava 0% de ausentes e 0 duplicados. Como não é uma
base real ainda, isso é esperado (é uma limitação conhecida, não uma qualidade real
comprovada) — mas confirmamos aqui, após as transformações acima, que nada foi
introduzido acidentalmente.


In [9]:
print("Ausentes por coluna (após transformações):")
print(df.isna().sum())
print("\nLinhas totalmente duplicadas:", df.duplicated().sum())


Ausentes por coluna (após transformações):
idade                      0
objetivo                   0
nivel_experiencia          0
dias_disponiveis_semana    0
restricao_fisica           0
equipamento_disponivel     0
treino_recomendado         0
dtype: int64

Linhas totalmente duplicadas: 1


### 3.5 Atualizar o dicionário de dados quanto à faixa de idade

**Motivo:** o dicionário de dados (Atividade 4) registrava a faixa de idade como
"16 a 65 anos", mas a base observada vai de 16 a 51 anos. Não há erro nos dados —
a amostra simulada simplesmente não gerou valores acima de 51. **Decisão:** não alterar
os dados (não é um erro a ser corrigido), e sim atualizar o dicionário de dados no
README para refletir a faixa realmente observada (16–51), evitando expectativa
incorreta sobre o alcance da base.

**Estratégia alternativa considerada:** forçar artificialmente alguns registros para
a faixa 52–65, para "bater" com o dicionário original. Descartada por inventar dados
que não existem na amostra real — o dicionário é a documentação que deve seguir o
dado, não o contrário.


In [10]:
print("Faixa de idade confirmada na base tratada:",
      df["idade"].min(), "a", df["idade"].max())


Faixa de idade confirmada na base tratada: 16 a 50


### 3.6 Ruído de rótulo (8%) — mantido, não removido

**Motivo:** o diagnóstico identificou que ~8% dos registros (24 linhas) têm o rótulo
`treino_recomendado` em desacordo com a regra de geração original. Como não existe
nenhum critério externo confiável para separar "ruído real" de "exceção legítima" —
ambos são estatisticamente idênticos do ponto de vista dos dados —, **a decisão é
manter todos os registros como estão**, sem tentar "corrigir" o rótulo.

**Estratégia alternativa considerada:** remover os registros suspeitos de ruído.
Descartada porque (a) não há forma confiável de identificá-los sem acesso à regra
de geração original em produção, e (b) removê-los arbitrariamente reduziria a base
já pequena (300 registros) e poderia introduzir viés de seleção. Isso é registrado
como uma **limitação assumida do projeto**, já documentada na retrospectiva
(Atividade 6).


### 3.7 Desbalanceamento leve entre classes — não tratado nesta etapa

**Motivo:** o diagnóstico apontou desbalanceamento leve (~1,7:1 entre a classe mais e
a menos frequente). **Decisão:** este não é um problema de limpeza de dados, e sim uma
decisão de modelagem — será tratado no momento do treinamento (Encontro 9), usando
divisão estratificada (`stratify`) na separação treino/teste, conforme já planejado na
Atividade 7. Nenhuma linha é duplicada, removida ou reponderada aqui.


## 4. Tabela de decisões

In [11]:
tabela_decisoes = pd.DataFrame([
    {
        "transformacao": "Remover aluno_id das features",
        "coluna_afetada": "aluno_id",
        "motivo": "Identificador sequencial sem relação com o perfil; risco de vazamento indireto",
        "impacto": "0 linhas removidas; 1 coluna passa de feature para índice",
        "risco": "Baixo — apenas reorganização, nenhuma informação de perfil é perdida",
    },
    {
        "transformacao": "Padronizar nomenclatura (strip + title case)",
        "coluna_afetada": "objetivo, nivel_experiencia, restricao_fisica, equipamento_disponivel, treino_recomendado",
        "motivo": "Inconsistência de nomenclatura identificada no diagnóstico (Atividade 5)",
        "impacto": "0 linhas removidas; nenhuma coluna removida; valores padronizados",
        "risco": "Baixo — transformação puramente sintática, não estatística",
    },
    {
        "transformacao": "Converter ordinais para categoria ordenada",
        "coluna_afetada": "nivel_experiencia, equipamento_disponivel",
        "motivo": "Variáveis ordinais estavam como texto nominal solto (inconsistência do diagnóstico)",
        "impacto": "0 linhas removidas; tipo de 2 colunas alterado de object para category ordenada",
        "risco": "Baixo — não altera valores, só a representação/ordem",
    },
    {
        "transformacao": "Confirmar ausência de nulos/duplicados",
        "coluna_afetada": "todas",
        "motivo": "Verificação de integridade após as transformações",
        "impacto": "0 linhas alteradas",
        "risco": "Nenhum — apenas verificação",
    },
    {
        "transformacao": "Atualizar dicionário de dados (faixa de idade)",
        "coluna_afetada": "idade (documentação, não os dados)",
        "motivo": "Dicionário dizia 16-65, base real vai de 16 a 51",
        "impacto": "0 linhas/colunas alteradas nos dados; dicionário no README será atualizado",
        "risco": "Nenhum nos dados; risco de documentação desatualizada se o README não for revisado",
    },
    {
        "transformacao": "Manter ruído de rótulo (8%) sem correção",
        "coluna_afetada": "treino_recomendado",
        "motivo": "Sem critério confiável para distinguir ruído de exceção legítima",
        "impacto": "0 linhas removidas/alteradas",
        "risco": "Médio — pode reduzir levemente a métrica do modelo; documentado como limitação assumida",
    },
    {
        "transformacao": "Não tratar desbalanceamento nesta etapa",
        "coluna_afetada": "treino_recomendado",
        "motivo": "Decisão de modelagem, não de limpeza; será tratada via stratify no split (Atividade 7)",
        "impacto": "0 linhas removidas/alteradas",
        "risco": "Baixo — mitigação já planejada para a etapa de treinamento",
    },
])

tabela_decisoes


,transformacao,coluna_afetada,motivo,impacto,risco
0,Remover aluno_id das features,aluno_id,Identificador sequencial sem relação com o perfil; risco de vazamento indireto,0 linhas removidas; 1 coluna passa de feature para índice,"Baixo — apenas reorganização, nenhuma informação de perfil é perdida"
1,Padronizar nomenclatura (strip + title case),"objetivo, nivel_experiencia, restricao_fisica, equipamento_disponivel, treino_recomendado",Inconsistência de nomenclatura identificada no diagnóstico (Atividade 5),0 linhas removidas; nenhuma coluna removida; valores padronizados,"Baixo — transformação puramente sintática, não estatística"
2,Converter ordinais para categoria ordenada,"nivel_experiencia, equipamento_disponivel",Variáveis ordinais estavam como texto nominal solto (inconsistência do diagnóstico),0 linhas removidas; tipo de 2 colunas alterado de object para category ordenada,"Baixo — não altera valores, só a representação/ordem"
3,Confirmar ausência de nulos/duplicados,todas,Verificação de integridade após as transformações,0 linhas alteradas,Nenhum — apenas verificação
4,Atualizar dicionário de dados (faixa de idade),"idade (documentação, não os dados)","Dicionário dizia 16-65, base real vai de 16 a 51",0 linhas/colunas alteradas nos dados; dicionário no README será atualizado,Nenhum nos dados; risco de documentação desatualizada se o README não for revisado
5,Manter ruído de rótulo (8%) sem correção,treino_recomendado,Sem critério confiável para distinguir ruído de exceção legítima,0 linhas removidas/alteradas,Médio — pode reduzir levemente a métrica do modelo; documentado como limitação assumida
6,Não tratar desbalanceamento nesta etapa,treino_recomendado,"Decisão de modelagem, não de limpeza; será tratada via stratify no split (Atividade 7)",0 linhas removidas/alteradas,Baixo — mitigação já planejada para a etapa de treinamento


## 5. Comparação antes / depois

In [12]:
print("ANTES:")
print(f"  Linhas: {df_original.shape[0]}  |  Colunas: {df_original.shape[1]}")
print(f"  Colunas: {list(df_original.columns)}")

print("\nDEPOIS:")
print(f"  Linhas: {df.shape[0]}  |  Colunas: {df.shape[1]} (+ aluno_id como índice)")
print(f"  Colunas: {list(df.columns)}")


ANTES:
  Linhas: 300  |  Colunas: 8
  Colunas: ['aluno_id', 'idade', 'objetivo', 'nivel_experiencia', 'dias_disponiveis_semana', 'restricao_fisica', 'equipamento_disponivel', 'treino_recomendado']

DEPOIS:
  Linhas: 300  |  Colunas: 7 (+ aluno_id como índice)
  Colunas: ['idade', 'objetivo', 'nivel_experiencia', 'dias_disponiveis_semana', 'restricao_fisica', 'equipamento_disponivel', 'treino_recomendado']


## 6. Verificação de vazamento de resposta

Checagem explícita exigida para esta entrega:


In [13]:
print("1) Alguma transformação aprendeu parâmetros sobre a base inteira antes do split?")
print("   -> NAO. As transformações aplicadas (remover coluna, padronizar texto,")
print("      converter tipo para categoria ordenada) sao deterministicas e nao")
print("      aprendem nenhuma estatistica dos dados (nao ha normalizacao, imputacao")
print("      por media/mediana, nem encoding que aprenda frequencias). Portanto nao ha")
print("      necessidade de refazer nada apos o split treino/teste (Atividade 7).")
print()
print("2) Identificadores foram removidos das features?")
print("   -> SIM. aluno_id foi movido para indice do DataFrame e nao integra")
print("      as colunas de entrada do modelo.")
print()
print("3) Alguma coluna que so existe apos o desfecho permanece entre as entradas?")
print("   -> NAO. Todas as 6 features (idade, objetivo, nivel_experiencia,")
print("      dias_disponiveis_semana, restricao_fisica, equipamento_disponivel)")
print("      descrevem o perfil do aluno ANTES de qualquer treino ser recomendado.")
print("      Apenas 'treino_recomendado' e posterior ao perfil, e e o label, nao uma feature.")


1) Alguma transformação aprendeu parâmetros sobre a base inteira antes do split?
   -> NAO. As transformações aplicadas (remover coluna, padronizar texto,
      converter tipo para categoria ordenada) sao deterministicas e nao
      aprendem nenhuma estatistica dos dados (nao ha normalizacao, imputacao
      por media/mediana, nem encoding que aprenda frequencias). Portanto nao ha
      necessidade de refazer nada apos o split treino/teste (Atividade 7).

2) Identificadores foram removidos das features?
   -> SIM. aluno_id foi movido para indice do DataFrame e nao integra
      as colunas de entrada do modelo.

3) Alguma coluna que so existe apos o desfecho permanece entre as entradas?
   -> NAO. Todas as 6 features (idade, objetivo, nivel_experiencia,
      dias_disponiveis_semana, restricao_fisica, equipamento_disponivel)
      descrevem o perfil do aluno ANTES de qualquer treino ser recomendado.
      Apenas 'treino_recomendado' e posterior ao perfil, e e o label, nao uma feature.


## 7. Salvar e verificar a base tratada

In [14]:
df.to_csv("dataset_academia_tratado.csv")

# Verificacao: reler o arquivo salvo e conferir shape/tipos
df_verificacao = pd.read_csv("dataset_academia_tratado.csv", index_col="aluno_id")
print("Base tratada salva e recarregada com sucesso.")
print(f"Linhas: {df_verificacao.shape[0]}  |  Colunas: {df_verificacao.shape[1]}")
df_verificacao.head()


Base tratada salva e recarregada com sucesso.
Linhas: 300  |  Colunas: 7


,idade,objetivo,nivel_experiencia,dias_disponiveis_semana,restricao_fisica,equipamento_disponivel,treino_recomendado
aluno_id,,,,,,,
1,32,Emagrecimento,Iniciante,5,Nenhuma,Limitado,Treino D
2,37,Hipertrofia,Intermediario,4,Joelho,Completo,Treino D
3,24,Condicionamento,Iniciante,2,Ombro,Limitado,Treino D
4,31,Emagrecimento,Iniciante,5,Nenhuma,Limitado,Treino D
5,23,Hipertrofia,Intermediario,2,Ombro,Completo,Treino B


In [15]:
import os
print("Arquivo original intacto:", os.path.exists("dataset_academia.csv"))
print("Linhas no original:", pd.read_csv("dataset_academia.csv").shape[0])


Arquivo original intacto: True
Linhas no original: 300


## 8. Prontidão para o Encontro 9

**O que já está pronto:**
- Base sem identificador entre as features, sem inconsistência de nomenclatura, com
  variáveis ordinais devidamente tipadas, e sem nenhum vazamento de resposta identificado.
- Base tratada salva em `dataset_academia_tratado.csv`, verificada por releitura.

**O que ainda falta antes do treinamento (Encontro 9):**
- Codificação numérica das variáveis categóricas (ex.: ordinal encoding para
  `nivel_experiencia`/`equipamento_disponivel`, one-hot para `objetivo` e
  `restricao_fisica`) — deixada para o Encontro 9 porque, para evitar vazamento,
  esse encoding deve ser ajustado (`fit`) apenas no conjunto de treino, depois do
  split, e não sobre a base inteira.
- A divisão treino/teste em si (80/20, estratificada), planejada na Atividade 7, ainda
  será executada no Encontro 9, imediatamente antes do treinamento do primeiro modelo.
